# Remediation · Fix `fact_notification.alert_category` (F1+F2)

**Why this exists.** EDA on 2026-04-30 found two related bugs:
1. `fact_notification.alert_category` was 'other' for all 659,566 rows because
   the loader derived it from `staging.notification.description` (100% NULL).
   The actual alert kind lives in `staging.notification.name`.
2. `mart_device_monthly_behavior.speed_alert_count` was 0 for every row
   because it was sourced from the empty `fact_speed_notification`.

**Fixes applied to SQL:**
- `sql/17_fact_notification_incr.sql` — categorize on `name`, not `description`.
- `sql/20_mart_device_monthly_behavior.sql` — alert CTE now reads from
  `warehouse.fact_notification WHERE alert_category='speed_alert'`.

**What this notebook does:**
1. Wipe and reload `warehouse.fact_notification` (UPSERT-on-key would only
   touch new rows; we need every existing row re-categorized).
2. Recompute `mart_device_monthly_behavior` and `mart_fleet_daily` so they
   pick up the new alert categories.
3. Verify with the post-fix queries from the plan.

In [1]:
from __future__ import annotations
import sys, pathlib, time
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for c in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = c / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src)); break

from accent_fleet.config import settings, load_pipeline_config
from accent_fleet.db import get_engine, run_sql_file, transaction
from accent_fleet.pipeline.run_log import begin_run, end_run
from sqlalchemy import text
from datetime import datetime, timedelta
import pandas as pd

cfg = load_pipeline_config()
MIN_TIME = datetime.fromisoformat(cfg['window']['min_event_time'].replace('Z','+00:00')).replace(tzinfo=None)
CHUNK_DAYS = cfg['window']['backfill_chunk_days']

## 2. Pre-fix audit — confirm the bug

In [2]:
with get_engine().connect() as conn:
    before = pd.read_sql(text('''
        SELECT alert_category, COUNT(*) AS n
        FROM warehouse.fact_notification
        GROUP BY 1 ORDER BY 2 DESC
    '''), conn)
before

,alert_category,n
0,other,659566


## 3. Wipe and reload `fact_notification`

`UPSERT ... DO UPDATE` on the existing rows would re-categorize them, but the
SQL is in INSERT-shape — the existing rows are already in place and the WHERE
clause only touches the window. The UPSERT branch *will* fire on conflicts and
set the new `description` and `alert_category`. So we just re-run the loader
across the full history and let the UPSERT do the recategorization.

In [3]:
import importlib
import accent_fleet.transforms.facts as facts_module
facts_module = importlib.reload(facts_module)
load_fact_incremental = facts_module.load_fact_incremental

# Reset the watermark so the loader re-processes the full history.
with transaction() as conn:
    conn.execute(text('''
        UPDATE warehouse.etl_watermark
        SET last_event_time = NULL, rows_loaded_total = 0
        WHERE table_name = 'fact_notification'
    '''))

with get_engine().connect() as conn:
    end_time = conn.execute(text("SELECT MAX(created_at) FROM staging.notification")).scalar_one()
print('reloading up to', end_time)

run_id = begin_run(mode='notebook:11_fix_alert_categories')
progress, cursor, t0 = [], MIN_TIME, time.time()
try:
    while cursor < end_time:
        nxt = min(cursor + timedelta(days=CHUNK_DAYS), end_time)
        res = load_fact_incremental('fact_notification', run_id=run_id, window_end=nxt)
        progress.append({'window_end': nxt, 'rows_loaded': res.rows_loaded})
        cursor = nxt
    end_run(run_id, status='success', rows_loaded=sum(p['rows_loaded'] for p in progress))
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc)); raise
print(f'done in {time.time()-t0:.1f}s; chunks={len(progress)}')

reloading up to 2026-04-10 13:32:24
2026-04-30 04:42:50 [info     ] fact_load.start                end=datetime.datetime(2019, 10, 31, 0, 0) fact=fact_notification run_id=37 start=datetime.datetime(2019, 10, 1, 0, 0)
2026-04-30 04:42:51 [info     ] fact_load.done                 fact=fact_notification new_watermark=datetime.datetime(2019, 10, 31, 0, 0) rows=317
2026-04-30 04:42:52 [info     ] fact_load.start                end=datetime.datetime(2019, 11, 30, 0, 0) fact=fact_notification run_id=37 start=datetime.datetime(2019, 10, 30, 23, 50)
2026-04-30 04:42:52 [info     ] fact_load.done                 fact=fact_notification new_watermark=datetime.datetime(2019, 11, 30, 0, 0) rows=432
2026-04-30 04:42:53 [info     ] fact_load.start                end=datetime.datetime(2019, 12, 30, 0, 0) fact=fact_notification run_id=37 start=datetime.datetime(2019, 11, 29, 23, 50)
2026-04-30 04:42:54 [info     ] fact_load.done                 fact=fact_notification new_watermark=datetime.datetime(201

## 4. Post-fix audit — categories should now be diverse

In [4]:
with get_engine().connect() as conn:
    after = pd.read_sql(text('''
        SELECT alert_category, COUNT(*) AS n
        FROM warehouse.fact_notification
        GROUP BY 1 ORDER BY 2 DESC
    '''), conn)
after

,alert_category,n
0,speed_alert,585218
1,maintenance_alert,73242
2,geofence_alert,685
3,fuel_theft_alert,418
4,ignition_alert,3


## 5. Recompute the affected marts

`mart_device_monthly_behavior` (sql/20) now sources speed alerts from
`fact_notification`; `mart_fleet_daily` (sql/30) splits the alert categories.
Both must be recomputed so the changed CTEs take effect.

In [5]:
with get_engine().connect() as conn:
    months = pd.read_sql(text('''
        SELECT DISTINCT TO_CHAR(created_at, 'YYYY-MM') AS m
        FROM warehouse.fact_notification
        ORDER BY 1
    '''), conn)['m'].tolist()
    dates = pd.read_sql(text('''
        SELECT DISTINCT notification_date AS d FROM warehouse.fact_notification
        ORDER BY 1
    '''), conn)['d'].astype(str).tolist()
print(f'months={len(months)} dates={len(dates)}')

run_id = begin_run(mode='notebook:11_fix_alert_categories.recompute')
try:
    with transaction() as conn:
        run_sql_file(conn, '20_mart_device_monthly_behavior.sql',
                     params={'touched_months': months, 'etl_run_id': run_id})
        run_sql_file(conn, '30_mart_fleet_daily.sql',
                     params={'touched_dates': dates, 'etl_run_id': run_id})
    end_run(run_id, status='success')
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc)); raise
print('marts recomputed.')

months=79 dates=2371
marts recomputed.


## 6. Final verification (the queries from the plan)

In [6]:
with get_engine().connect() as conn:
    v1 = conn.execute(text('''
        SELECT COUNT(*) FROM warehouse.fact_notification
        WHERE alert_category = 'speed_alert'
    ''')).scalar_one()
    v2 = conn.execute(text('''
        SELECT COUNT(*) FILTER (WHERE speed_alert_count > 0)
        FROM marts.mart_device_monthly_behavior
    ''')).scalar_one()
    v3 = conn.execute(text('''
        SELECT ROUND(100.0*COUNT(*) FILTER (WHERE speed_alert_count > 0) / COUNT(*), 1)
        FROM marts.v_ml_features_full
        WHERE year_month >= '2025-01'
    ''')).scalar_one()
print(f'V1 fact_notification speed_alerts (target ~585,545): {v1:,}')
print(f'V2 mart rows with speed_alert_count > 0:            {v2:,}')
print(f'V3 modeling-window pct with speed alerts:           {v3}%')
assert v1 > 500_000, 'F1 did not unlock speed alerts'
assert v2 > 0,       'F2 did not propagate to the ML mart'
print('\nF1 + F2 verified.')

V1 fact_notification speed_alerts (target ~585,545): 585,218
V2 mart rows with speed_alert_count > 0:            2,213
V3 modeling-window pct with speed alerts:           18.3%

F1 + F2 verified.
